# Qwen3.5 2B local qua Ollama: sân thử tool calling

Gọi thẳng API native của Ollama (`/api/chat`) với `think: false`, nên model trả lời luôn, không có khối suy nghĩ.
Cell 3 là mock: 2 tool giả, 1 câu lệnh, **1 lượt gọi model**. Sửa `USER_TEXT` hoặc `TOOLS` rồi chạy lại để thử.
Cần Ollama đang chạy (`ollama serve`) và đã có model (`ollama list` thấy `qwen3.5:2b`).

Lưu ý: đây là để thử nhanh ở máy; model này không phải model của bài nộp (Qwen3-4B, chạy trên Kaggle).

In [ ]:
import json, time, requests

OLLAMA = "http://localhost:11434"
MODEL = "qwen3.5:2b"
OPTIONS = {"temperature": 0, "seed": 0}   # greedy, lặp lại được


def chat(messages, tools=None, think=False, model=None, fmt=None, **options):
    """Một lượt gọi model. Trả về (message, thống kê). think=False tắt thinking; model=None dùng MODEL;
    fmt="json" ép Ollama trả JSON hợp lệ."""
    body = {"model": model or MODEL, "messages": messages, "stream": False, "think": think,
            "options": {**OPTIONS, **options}}
    if fmt:
        body["format"] = fmt
    if tools:
        body["tools"] = tools
    t = time.time()
    r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=300)
    r.raise_for_status()
    d = r.json()
    stats = {"giây": round(time.time() - t, 2), "token vào": d.get("prompt_eval_count"), "token ra": d.get("eval_count")}
    return d["message"], stats


def show(msg, stats):
    print("thống kê:", stats)
    print("thinking:", repr(msg.get("thinking")) if msg.get("thinking") else "(không có, đã tắt)")
    print("content :", repr(msg.get("content")))
    for c in msg.get("tool_calls") or []:
        f = c["function"]
        print("tool_call:", f["name"], json.dumps(f["arguments"], ensure_ascii=False))
    if not msg.get("tool_calls"):
        print("tool_call: (không gọi tool)")


print([m["name"] for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]])

## Mock: 2 tool giả + 1 lượt gọi

Schema viết theo đúng kiểu catalog của cuộc thi (`contacts.search`, `phone.call`, xem bằng `scripts/show_tool.py`).
`MOCK_RESULTS` là kết quả giả mà tool trả về, chỉ dùng khi bạn muốn chạy thêm lượt 2 ở cell cuối.

In [ ]:
SYSTEM = ("Bạn là trợ lý điện thoại. Chỉ gọi tool có trong danh sách. "
          "Cần số hay ID liên hệ thì tra contacts.search trước, không tự bịa số.")
USER_TEXT = "Gọi cho Bố giúp tôi nha"

TOOLS = [
    {"type": "function", "function": {
        "name": "contacts.search",
        "description": "Tìm liên hệ theo tên, quan hệ hoặc số điện thoại",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Tên, quan hệ (mẹ, vợ, sếp) hoặc số điện thoại"}}, "required": []}}},
    {"type": "function", "function": {
        "name": "phone.call",
        "description": "Gọi thoại cho liên hệ hoặc số điện thoại. Bắt buộc một trong: contact_id hoặc phone_number",
        "parameters": {"type": "object", "properties": {
            "contact_id": {"type": "string", "description": "ID liên hệ, lấy từ contacts.search"},
            "phone_number": {"type": "string", "description": "Số điện thoại, giữ nguyên chuỗi"}}, "required": []}}},
]
MOCK_RESULTS = {"contacts.search": {"ok": True, "contacts": [{"id": "c_01", "name": "Bố", "phone": "0912000111"}]},
                "phone.call": {"ok": True, "status": "calling"}}

messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": USER_TEXT}]
msg, stats = chat(messages, TOOLS)
show(msg, stats)

## (Tuỳ chọn) Lượt 2: trả kết quả mock về cho model

Chạy cell này nếu muốn xem model có đi tiếp (ví dụ sau `contacts.search` thì gọi `phone.call` với `contact_id`) hay không.

In [ ]:
calls = msg.get("tool_calls") or []
if calls:
    messages2 = messages + [msg] + [
        {"role": "tool", "tool_name": c["function"]["name"],
         "content": json.dumps(MOCK_RESULTS.get(c["function"]["name"], {"ok": True}), ensure_ascii=False)} for c in calls]
    msg2, stats2 = chat(messages2, TOOLS)
    show(msg2, stats2)
else:
    print("Lượt 1 không gọi tool, không có gì để trả về.")

## Phân loại nhóm tool bằng `tev1:0.8b`

`tev1:0.8b` là model họ qwen35, 752M tham số, Q8_0, `num_ctx` 2050.
System prompt mặc định của nó là bài chọn MỘT đáp án và trả về một chữ cái; ở đây system prompt của mình (prompt của bạn, giữ nguyên) thay thế nó.
Thinking tắt (`think=False`).

In [ ]:
CLS_MODEL = "tev1:0.8b"

messages = [
{
    'role' : 'system',
    'content' : """
        You are a tool group classifier.

        Your task is to select ALL relevant tool groups needed to fulfill the user's query.

        Available groups:
        - smart_home: Smart devices, security, door locks, appliances, home automation.
        - time_management: Alarms, reminders, calendar, notes, tasks.
        - communication: Contacts, phone, SMS, OTT chat, email, video meetings.
        - finance: Banking, e-wallets, bill payments, loyalty, subscriptions.
        - information: Weather, prices, exchange rates, news, calculator.
        - transportation: Maps, ride-hailing, flights, bike rentals.
        - shopping: Food delivery, shopping lists, package tracking.
        - entertainment: Music, TV, podcasts, radio.
        - health: Medication, fitness.
        - learning: Learning resources, library.

        IMPORTANT RULES:
        1. A query CAN require MULTIPLE groups.
        2. Always check whether the query contains additional actions belonging to other groups.
        3. Do NOT stop after finding the first matching group.
        4. Select every group directly needed to complete the requested actions.
        5. Do NOT select unrelated groups.
        6. If multiple actions belong to the same group, return that group only ONCE.
        7. If no tool is needed, return an empty groups list.
        8. Use only the available group names.
        9. Return valid JSON only. No additional text.

        Examples:

        User: "Đặt báo thức lúc 6 giờ sáng và nhắc tôi uống thuốc lúc 8 giờ."
        Output:
        {"groups": ["alarm", "reminders", "medication"]}

        User: "Kiểm tra thời tiết Hà Nội ngày mai rồi đặt xe đến sân bay."
        Output:
        {"groups": ["weather", "ride_hailing", "maps"]}

        User: "Gửi email cho giảng viên và tạo lịch họp vào thứ Hai."
        Output:
        {"groups": ["email", "calendar"]}

        User: "Tắt đèn phòng khách và khóa cửa trước khi đi ngủ."
        Output:
        {"groups": ["smart_home", "door_lock"]}

        User: "Tìm giá vàng hôm nay và đổi 100 USD sang tiền Việt."
        Output:
        {"groups": ["gold_price", "exchange_rate", "calculator"]}

        User: "Phát nhạc và đặt báo thức lúc 6 giờ sáng."
        Output:
        {"groups": ["music", "alarm"]}

        User: "Hôm nay thời tiết thế nào?"
        Output:
        {"groups": ["weather"]}

        User: "Tôi muốn học Python."
        Output:
        {"groups": ["learning"]}

        Now classify this query:

        Query:

            """
},
{
    'role' : 'user',
    'content' : """
      Đặt mục tiêu tám ngàn bước mỗi ngày rồi nhắc tôi uống nước lúc 3 giờ chiều.
    """
}
]
SYSTEM_CLS = messages[0]["content"]
msg , stats = chat(messages, model=CLS_MODEL)
show(msg, stats)

### Đo trên mẫu train

Gold = nhóm của các app mà gold gọi (`sys.*` không tính), quy về 10 nhóm của prompt.
Nếu model trả tên app (như các ví dụ trong prompt) thì cũng quy về nhóm, để chấm công bằng; cột `tên lạ` đếm các tên không phải nhóm cũng không phải app.
Chỉ dùng câu cuối (không lịch sử), giống ô ở trên. Ô này in id và nhóm, không in câu (repo public: đừng commit output có câu của mẫu).

In [ ]:
import sys, random, re
sys.path.insert(0, "..")
from lifemate.data import load_samples

GROUP_APPS = {
    "smart_home": "smart_home home_security door_lock robot_vacuum appliances home_energy home_automation household_chores",
    "time_management": "alarm reminders calendar notes tasks",
    "communication": "contacts phone sms chat_ott email video_meeting",
    "finance": "banking ewallet bill_payment loyalty subscriptions",
    "information": "weather gold_price fuel_price lottery exchange_rate news calculator",
    "transportation": "maps ride_hailing flights bike_rental",
    "shopping": "food_delivery shopping_list package_tracking",
    "entertainment": "music tv podcast_radio",
    "health": "medication fitness",
    "learning": "learning library",
}
APP2G = {a: g for g, s in GROUP_APPS.items() for a in s.split()}


def gold_groups(s):
    return {APP2G[c["name"].split(".")[0]] for c in s["label"]["tool_calls"] if not c["name"].startswith("sys.")}


def parse_groups(text):
    """JSON {"groups": [...]} -> (set of groups, list of unknown names, parsed ok?)."""
    m = re.search(r"\{.*\}", text or "", re.S)
    try:
        names = json.loads(m.group(0))["groups"] if m else None
    except Exception:
        names = None
    if not isinstance(names, list):
        return set(), [], False
    got, odd = set(), []
    for n in names:
        n = str(n).strip()
        if n in GROUP_APPS:
            got.add(n)
        elif n in APP2G:
            got.add(APP2G[n])
        else:
            odd.append(n)
    return got, odd, True


N = 40                       # số mẫu thử; tăng dần
random.seed(0)
pool = random.sample(load_samples("train"), N)
rows = []
for s in pool:
    m, st = chat([{"role": "system", "content": SYSTEM_CLS}, {"role": "user", "content": s["audio_text"]}], model=CLS_MODEL)
    pred, odd, ok = parse_groups(m.get("content"))
    gold = gold_groups(s)
    rows.append({"id": s["id"], "gold": gold, "pred": pred, "odd": odd, "ok": ok, "sec": st["giây"]})
    print(f'{s["id"]}  gold {sorted(gold)}  pred {sorted(pred)}' + (f"  tên lạ {odd}" if odd else "") + ("" if ok else "  [không parse được]"))

tp = sum(len(r["gold"] & r["pred"]) for r in rows)
fp = sum(len(r["pred"] - r["gold"]) for r in rows)
fn = sum(len(r["gold"] - r["pred"]) for r in rows)
p, rc = tp / max(1, tp + fp), tp / max(1, tp + fn)
print(f"\n{N} mẫu | đúng hẳn tập nhóm {sum(r['gold'] == r['pred'] for r in rows)}/{N}"
      f" | phủ đủ gold (recall mẫu) {sum(r['gold'] <= r['pred'] for r in rows)}/{N}"
      f" | precision {p:.1%} recall {rc:.1%} F1 {2 * p * rc / max(1e-9, p + rc):.1%}"
      f" | JSON hỏng {sum(not r['ok'] for r in rows)} | có tên lạ {sum(bool(r['odd']) for r in rows)}"
      f" | {sum(r['sec'] for r in rows) / N:.2f} s/mẫu")